# **FaceModeExplorer: *Video Face Analysis Using InsightFace and HSEmotion***

## Project Overview

This project is an AI-powered facial analysis system that combines **InsightFace** and **HSEmotion** to analyze video content and extract demographic and emotional information from detected faces.

The system first detects faces in video frames using **InsightFace**, which also estimates the **age** and **gender** of each detected face. The detected face regions are then analyzed using **HSEmotion** to recognize the person's facial emotion.

The extracted predictions are aggregated to generate an annotated output video, an interactive analytics dashboard, and a downloadable CSV report, providing clear visual and statistical insights from the analyzed video.

## Import Libraries

In [1]:
import os
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
from insightface.app import FaceAnalysis
from hsemotion.facial_emotions import HSEmotionRecognizer
from google.colab.patches import cv2_imshow

## Load Models

In [2]:
# Select execution device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"▶ Using Device: {DEVICE}")

# Initialize InsightFace model
face_app = FaceAnalysis(name="buffalo_l", allowed_modules=["detection", "genderage"])

# GPU = 0 , CPU = -1
ctx_id = 0 if DEVICE == "cuda" else -1

face_app.prepare(ctx_id=ctx_id, det_thresh=0.5)

print("InsightFace model loaded")

# Initialize HSEmotion model
emotion_recognizer = HSEmotionRecognizer(model_name="enet_b0_8_best_vgaf", device=DEVICE)

print("HSEmotion model loaded")

▶ Using Device: cpu
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
model ignore: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition
set det-size: [(128, 128), (640, 640)]
InsightFace model loaded
/root/.hsemotion/enet_b0_8_best_vgaf.pt Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
  

/usr/local/lib/python3.12/dist-packages/hsemotion/facial_emotions.py:49: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model=torch.load(path)


## Face Detection Function

In [3]:
#Detect faces using InsightFace and return face objects.
def detect_faces(frame):

    faces = face_app.get(frame)

    return faces

## Age and Gender Prediction

In [4]:
#Predict age and gender using InsightFace.
def predict_age_gender(face):

    age = int(face.age)

    raw_gender = getattr(face, "sex", getattr(face, "gender", None))
    val = str(raw_gender).strip().upper()

    if val in ["1", "M", "MALE"]:
        gender = "Male"
    elif val in ["0", "F", "FEMALE"]:
        gender = "Female"
    else:
        gender = "Unknown"

    return age, gender

## Emotion Prediction



In [5]:
#Predict the dominant emotion using HSEmotion.
def predict_emotion(frame, face):

    # Get face bounding box
    x1, y1, x2, y2 = face.bbox.astype(int)

    # Keep coordinates inside image boundaries
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(frame.shape[1], x2)
    y2 = min(frame.shape[0], y2)

    # Crop the face region
    face_crop = frame[y1:y2, x1:x2]

    # Return Unknown if the crop is invalid
    if face_crop.size == 0:
        return "Unknown"

    # Predict emotion
    emotion, _ = emotion_recognizer.predict_emotions(face_crop)

    return emotion

## Result Drawing

In [6]:
#Draw face detection results on the current frame
def draw_results(frame, faces, age_gender_results, emotion_results):

    PURPLE = (180, 50, 255)   # BGR

    for face, (age, gender), emotion in zip(faces, age_gender_results, emotion_results):

        x1, y1, x2, y2 = face.bbox.astype(int)

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(frame.shape[1], x2)
        y2 = min(frame.shape[0], y2)

        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), PURPLE, 2)

        label = (
            f"Age: {age} | "
            f"{gender} | "
            f"{emotion}"
        )

        # Background for text
        (text_width, text_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)

        cv2.rectangle(frame, (x1, y1 - 35), (x1 + text_width + 10, y1), PURPLE, -1)

        cv2.putText(frame, label, (x1 + 5, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)

    return frame

## Video Processing

In [7]:
def analyze_video(video_path, frame_skip=5):

    # Open video
    cap = cv2.VideoCapture(video_path)

    # Store predictions for summary statistics
    ages = []
    genders = []
    emotions = []

    # Store the latest analysis results
    last_faces = []
    last_age_gender = []
    last_emotions = []

    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Output video path
    output_path = "/content/annotated_output.mp4"

    # Create video writer
    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    frame_count = 0

    while True:

        # Read next frame
        ret, frame = cap.read()

        # Stop when video ends
        if not ret:
            break

        frame_count += 1

        # Analyze every N frames
        if frame_count % frame_skip == 0:

            faces = detect_faces(frame)

            current_age_gender = []
            current_emotions = []

            for face in faces:

                age, gender = predict_age_gender(face)
                emotion = predict_emotion(frame, face)

                current_age_gender.append((age, gender))
                current_emotions.append(emotion)

                ages.append(age)
                genders.append(gender)
                emotions.append(emotion)

            # Save latest results
            last_faces = faces
            last_age_gender = current_age_gender
            last_emotions = current_emotions

        # Draw results on every frame
        frame = draw_results(
            frame,
            last_faces,
            last_age_gender,
            last_emotions
        )

        # Save frame
        writer.write(frame)

    # Release resources
    cap.release()
    writer.release()

    print(f"Annotated video saved to: {output_path}")

    return output_path, ages, genders, emotions


## Run Video Analysis

In [8]:
video_path = "/content/IMG_2436.MP4"

output_path, ages, genders, emotions = analyze_video(video_path)

Annotated video saved to: /content/annotated_output.mp4


## Summary Results

In [9]:
# Compact Interactive Dashboard

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from IPython.display import display, HTML
from collections import Counter
import pandas as pd

if ages:

    avg_age = int(sum(ages) / len(ages))

    gender_counts = Counter(genders)
    emotion_counts = Counter(emotions)

    final_gender = gender_counts.most_common(1)[0][0]
    final_emotion = emotion_counts.most_common(1)[0][0]
    total_predictions = len(ages)

    df = pd.DataFrame({
        "Age": ages,
        "Gender": genders,
        "Emotion": emotions
    })

    display(HTML(f"""
    <div style="
        background: linear-gradient(135deg, #12071F, #24113B, #3B1760);
        padding: 18px;
        border-radius: 18px;
        color: white;
        font-family: Arial;
        max-width: 1000px;
        margin: 0 auto 14px auto;
        box-shadow: 0 6px 18px rgba(0,0,0,0.30);
    ">
        <h2 style="margin:0; font-size:26px;">Face Analysis Dashboard</h2>
        <p style="color:#D8C7FF; font-size:13px; margin:6px 0 0 0;">
            AI-powered video analysis using InsightFace and HSEmotion
        </p>

        <div style="
            display:grid;
            grid-template-columns: repeat(4, 1fr);
            gap:10px;
            margin-top:15px;
        ">
            <div style="background:#26123D; padding:12px; border-radius:13px;">
                <p style="color:#C9A7FF; margin:0; font-size:13px;">Average Age</p>
                <h2 style="margin:6px 0 0 0; font-size:24px;">{avg_age:.2f}</h2>
            </div>

            <div style="background:#26123D; padding:12px; border-radius:13px;">
                <p style="color:#C9A7FF; margin:0; font-size:13px;">Final Gender</p>
                <h2 style="margin:6px 0 0 0; font-size:24px;">{final_gender}</h2>
            </div>

            <div style="background:#26123D; padding:12px; border-radius:13px;">
                <p style="color:#C9A7FF; margin:0; font-size:13px;">Final Emotion</p>
                <h2 style="margin:6px 0 0 0; font-size:24px;">{final_emotion}</h2>
            </div>

            <div style="background:#26123D; padding:12px; border-radius:13px;">
                <p style="color:#C9A7FF; margin:0; font-size:13px;">Total Detections</p>
                <h2 style="margin:6px 0 0 0; font-size:24px;">{total_predictions}</h2>
            </div>
        </div>
    </div>
    """))

    gender_df = pd.DataFrame({
        "Gender": list(gender_counts.keys()),
        "Count": list(gender_counts.values())
    })

    emotion_df = pd.DataFrame({
        "Emotion": list(emotion_counts.keys()),
        "Count": list(emotion_counts.values())
    }).sort_values(by="Count", ascending=False)

    fig = make_subplots(
        rows=1,
        cols=3,
        specs=[[{"type": "domain"}, {"type": "xy"}, {"type": "xy"}]],
        subplot_titles=(
            "Gender Distribution",
            "Emotion Distribution",
            "Age Distribution"
        )
    )

    fig.add_trace(
        go.Pie(
            labels=gender_df["Gender"],
            values=gender_df["Count"],
            hole=0.45,
            marker=dict(colors=["#7C3AED", "#C084FC", "#4C1D95"]),
            textinfo="label+percent"
        ),
        row=1,
        col=1
    )

    fig.add_trace(
        go.Bar(
            x=emotion_df["Emotion"],
            y=emotion_df["Count"],
            text=emotion_df["Count"],
            textposition="auto",
            marker_color="#A855F7"
        ),
        row=1,
        col=2
    )

    fig.add_trace(
        go.Histogram(
            x=df["Age"],
            nbinsx=10,
            marker_color="#8B5CF6"
        ),
        row=1,
        col=3
    )

    fig.update_layout(
        width=1000,
        height=390,
        paper_bgcolor="#12071F",
        plot_bgcolor="#12071F",
        font=dict(color="white", size=11),
        title=dict(
            text="Visual Summary",
            font=dict(size=20, color="#E9D5FF"),
            x=0.5
        ),
        showlegend=False,
        margin=dict(l=25, r=25, t=70, b=35)
    )

    fig.update_xaxes(color="white")
    fig.update_yaxes(color="white", gridcolor="#3B1760")

    fig.show()

    display(HTML("""
    <div style="
        max-width:1000px;
        margin: 12px auto 8px auto;
        font-family: Arial;
    ">
        <h3 style="color:#3B1760; margin-bottom:6px;">Detailed Results Table</h3>
    </div>
    """))

    display(df.head(20))

    csv_path = "/content/face_analysis_dashboard_data.csv"
    df.to_csv(csv_path, index=False)

    print(f"Dashboard data saved to: {csv_path}")
    print(f"Annotated video saved to: {output_path}")

else:
    display(HTML("""
    <div style="
        background:#26123D;
        padding:18px;
        border-radius:15px;
        color:white;
        font-family:Arial;
        max-width:900px;
        margin:auto;
    ">
        <h3>No faces were detected or analyzed.</h3>
    </div>
    """))

,Age,Gender,Emotion
0,28,Male,Happiness
1,29,Male,Happiness
2,30,Female,Neutral
3,29,Male,Happiness
4,32,Male,Happiness
5,38,Male,Happiness
6,30,Male,Happiness
7,27,Male,Happiness
8,31,Male,Happiness
9,32,Female,Neutral


Dashboard data saved to: /content/face_analysis_dashboard_data.csv
Annotated video saved to: /content/annotated_output.mp4


In [14]:
# ==========================================
# Gradio Interface
# ==========================================

import gradio as gr
import pandas as pd
from collections import Counter
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show_loading():
    loading_html = """
    <div class="result-card loading-card">
        <h2>Working on your video...</h2>
        <p>The analysis is running now. You can stay on this page while we prepare the results.</p>
        <div class="loader"></div>
    </div>
    """
    return (
        gr.update(visible=False),
        gr.update(visible=True),
        loading_html,
        None,
        None,
        pd.DataFrame(),
        gr.update(value=None, visible=False)
    )


def gradio_analyze(video_path):
    output_path, ages, genders, emotions = analyze_video(video_path)

    if not ages:
        summary = """
        <div class="result-card">
            <h2>No clear faces found</h2>
            <p>Try another video with better lighting or closer faces.</p>
        </div>
        """
        return (
            output_path,
            summary,
            None,
            pd.DataFrame(),
            gr.update(value=None, visible=False)
        )

    ages = [int(a) for a in ages]
    avg_age = int(round(sum(ages) / len(ages)))

    gender_counts = Counter(genders)
    emotion_counts = Counter(emotions)

    final_gender = gender_counts.most_common(1)[0][0]
    final_emotion = emotion_counts.most_common(1)[0][0]
    total_predictions = len(ages)

    df = pd.DataFrame({
        "Age": ages,
        "Gender": genders,
        "Emotion": emotions
    })

    csv_path = "/content/face_analysis_results.csv"
    df.to_csv(csv_path, index=False)

    summary = f"""
    <div class="result-card">
        <h2>Done, your video is ready</h2>
        <p>Here is a clean summary from the analyzed frames.</p>

        <div class="stats-grid">
            <div class="stat-box">
                <span>Average Age</span>
                <strong>{avg_age}</strong>
            </div>
            <div class="stat-box">
                <span>Most Seen Gender</span>
                <strong>{final_gender}</strong>
            </div>
            <div class="stat-box">
                <span>Main Expression</span>
                <strong>{final_emotion}</strong>
            </div>
            <div class="stat-box">
                <span>Face Detections</span>
                <strong>{total_predictions}</strong>
            </div>
        </div>
    </div>
    """

    gender_df = pd.DataFrame({
        "Gender": list(gender_counts.keys()),
        "Count": list(gender_counts.values())
    })

    emotion_df = pd.DataFrame({
        "Emotion": list(emotion_counts.keys()),
        "Count": list(emotion_counts.values())
    }).sort_values(by="Count", ascending=False)

    fig = make_subplots(
        rows=1,
        cols=3,
        specs=[[{"type": "domain"}, {"type": "xy"}, {"type": "xy"}]],
        subplot_titles=("Gender Mix", "Expressions", "Age Spread")
    )

    fig.add_trace(
        go.Pie(
            labels=gender_df["Gender"],
            values=gender_df["Count"],
            hole=0.55,
            marker=dict(colors=["#6D4C9F", "#B48AD8", "#8E6BBE"]),
            textinfo="label+percent"
        ),
        row=1,
        col=1
    )

    fig.add_trace(
        go.Bar(
            x=emotion_df["Emotion"],
            y=emotion_df["Count"],
            marker_color="#8E6BBE",
            text=emotion_df["Count"],
            textposition="auto"
        ),
        row=1,
        col=2
    )

    fig.add_trace(
        go.Histogram(
            x=df["Age"],
            nbinsx=10,
            marker_color="#6D4C9F"
        ),
        row=1,
        col=3
    )

    fig.update_layout(
        height=380,
        paper_bgcolor="#11101A",
        plot_bgcolor="#11101A",
        font=dict(color="#F5F1FF"),
        title=dict(
            text="Video Insights",
            x=0.5,
            font=dict(size=21, color="#E6D7FF")
        ),
        showlegend=False,
        margin=dict(l=20, r=20, t=65, b=25)
    )

    fig.update_xaxes(color="#F5F1FF", gridcolor="#34264A")
    fig.update_yaxes(color="#F5F1FF", gridcolor="#34264A")

    return (
        output_path,
        summary,
        fig,
        df,
        gr.update(value=csv_path, visible=True)
    )


def reset_app():
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        "",
        None,
        None,
        pd.DataFrame(),
        gr.update(value=None, visible=False)
    )


css = """
.gradio-container {
    background: linear-gradient(135deg, #090713, #15111F, #241633) !important;
    color: #F5F1FF !important;
}

#hero {
    text-align: center;
    padding: 22px 16px;
    border-radius: 20px;
    background: rgba(255, 255, 255, 0.04);
    border: 1px solid rgba(190, 160, 225, 0.15);
    margin-bottom: 16px;
}

#hero h1 {
    font-size: 38px;
    margin: 0 0 6px 0;
    color: #D9C2FF;
}

#hero p {
    font-size: 15px;
    color: #CFC3E8;
}

.upload-card, .result-card {
    padding: 16px;
    border-radius: 16px;
    background: rgba(20, 15, 32, 0.88);
    border: 1px solid rgba(190, 160, 225, 0.16);
}

.stats-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 10px;
    margin-top: 14px;
}

.stat-box {
    background: linear-gradient(135deg, #20172E, #2F2142);
    padding: 13px;
    border-radius: 14px;
    border: 1px solid rgba(190, 160, 225, 0.14);
}

.stat-box span {
    display: block;
    color: #CDB7EC;
    font-size: 12px;
}

.stat-box strong {
    display: block;
    margin-top: 6px;
    font-size: 22px;
    color: #FFFFFF;
}

button {
    border-radius: 14px !important;
    font-weight: 700 !important;
    background: linear-gradient(90deg, #6D4C9F, #9B6BCB) !important;
    border: none !important;
}

.loader {
    margin-top: 18px;
    width: 100%;
    height: 10px;
    border-radius: 20px;
    background: linear-gradient(90deg, #6D4C9F, #B48AD8, #6D4C9F);
    background-size: 200% 100%;
    animation: loadingMove 1.2s linear infinite;
}

@keyframes loadingMove {
    0% { background-position: 200% 0; }
    100% { background-position: -200% 0; }
}
"""


with gr.Blocks(css=css, theme=gr.themes.Soft(primary_hue="purple")) as demo:

    gr.HTML("""
    <div id="hero">
        <h1>Face Mood Explorer</h1>
        <p>Upload a video, run the analysis, and view everything in one clean results page.</p>
    </div>
    """)

    with gr.Group(visible=True) as upload_page:
        gr.HTML("""
        <div class="upload-card">
            <h2>Upload your video</h2>
            <p>Choose a clear short clip where faces are visible.</p>
        </div>
        """)
        input_video = gr.Video(label="Upload Video", height=330)
        analyze_btn = gr.Button("Start Analysis", variant="primary")

    with gr.Group(visible=False) as results_page:
        summary_output = gr.HTML()
        output_video = gr.Video(label="Annotated Video", height=360)
        dashboard_plot = gr.Plot(label="Video Insights")

        with gr.Accordion("Detailed Results and Downloads", open=False):
            results_table = gr.Dataframe(
                label="Detailed Results",
                interactive=False
            )
            download_btn = gr.DownloadButton(
                label="Download Results CSV",
                value=None,
                visible=False
            )

        reset_btn = gr.Button("Analyze Another Video")

    analyze_btn.click(
        fn=show_loading,
        inputs=None,
        outputs=[
            upload_page,
            results_page,
            summary_output,
            output_video,
            dashboard_plot,
            results_table,
            download_btn
        ]
    ).then(
        fn=gradio_analyze,
        inputs=input_video,
        outputs=[
            output_video,
            summary_output,
            dashboard_plot,
            results_table,
            download_btn
        ]
    )

    reset_btn.click(
        fn=reset_app,
        inputs=None,
        outputs=[
            upload_page,
            results_page,
            summary_output,
            output_video,
            dashboard_plot,
            results_table,
            download_btn
        ]
    )

demo.queue()
demo.launch(share=True)

/tmp/ipykernel_90067/1294266512.py:267: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_90067/1294266512.py:267: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1507217ca913dfeef9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
